### Section 1: What are Guardrails?
Guardrails help you build safe, compliant AI applications by validating and filtering content at key points in your agent's execution.

They are implemented as middleware that intercepts execution:

- Before the agent starts (input guardrails)
- After it completes (output guardrails)
- Around model and tool calls
Common Use Cases:
Use Case	Example
PII leakage prevention	Redact emails/credit cards before logging
Prompt injection blocking	Detect adversarial inputs
Harmful content filtering	Block dangerous requests
Business rule enforcement	Require approval for financial ops
Output quality validation	Ensure response meets safety standards


⚖️ Section 2: Two Approaches to Guardrails
Deterministic Guardrails
Rule-based: regex, keyword matching, explicit checks
- Fast, predictable, cost-effective
- May miss nuanced violations


Model-Based Guardrails
Uses LLMs/classifiers for semantic understanding
- Catches subtle/nuanced issues
- Slower and more expensive

In [6]:
import re

def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into database",
    "What is the capital of France",
    "Explain how malware spreads"
]

print("=== Deterministic Guardrail Demo ===")

for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
Blocked: How do I hack into database
Allowed: What is the capital of France
Blocked: Explain how malware spreads


In [7]:
from langchain_groq import ChatGroq

def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety."""

    model = ChatGroq(
    model="llama-3.3-70b-versatile"
)

    prompt = f"""
Is the following user input safe to process?

Reply with only 'Safe' or 'Unsafe'.

Input: {text}
"""

    result = model.invoke(prompt)
    return result.content.strip()


test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads."
]

print("=== Model-based Guardrail Demo ===")

for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if verdict.lower() == "unsafe" else "SAFE"
    print(f"{status}: {inp}")

=== Model-based Guardrail Demo ===
UNSAFE: How do I hack into a database?
SAFE: What is the capital of France?
SAFE: Explain how malware spreads.


### Section 3: Built-in Guardrail — PII Detection Middleware
LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

- Supported PII Types:
Type	Example
email	user@example.com
credit_card	5105-1051-0510-5100
ip	192.168.1.1
mac_address	00:1A:2B:3C:4D:5E
url	https://secret-site.com

- Strategies:
Strategy	Result
redact	[REDACTED_EMAIL]
mask	****-****-****-1234
hash	a8f5f167...
block	Raises an exception


In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# define a simple dummy tool 
@tool 
def customer_lookup(query: str) -> str:
    """ Look up customer information """
    return f"Customer record found for query:{query}"

    # Create agent with PII middleware 
    agent = create_agent(
        model = 'gpt-4o',
        tools=[customer_lookup],
        middleware=[
            # React emails in user input before sending to model 
            PIIMiddleware(
                "email",
                
            )
        ]
    )

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

# Initialize Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Create agent with PII Middleware
agent = create_agent(
    model=llm,
    tools=[customer_lookup],
    middleware=[
        # Redact emails
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),

        # Mask credit cards
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),

        # Block API keys
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [11]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is niko@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I've located your customer record. Is there something specific you'd like to know or discuss regarding your account?


In [12]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='d3ab1204-84ba-481f-a961-3a1d2246585e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q1xb51fs2', 'function': {'arguments': '{"query":"[REDACTED_EMAIL] and card ****-****-****-5100"}', 'name': 'customer_lookup'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 236, 'total_tokens': 266, 'completion_time': 0.073876334, 'completion_tokens_details': None, 'prompt_time': 0.012146708, 'prompt_tokens_details': None, 'queue_time': 0.162092051, 'total_time': 0.086023042}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019effa8-ae90-75c2-a217-93a33c67d6eb-0', tool_calls=[{'name': 'customer_lookup', 'args': {'quer

In [14]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content


In [15]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='d3ab1204-84ba-481f-a961-3a1d2246585e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q1xb51fs2', 'function': {'arguments': '{"query":"[REDACTED_EMAIL] and card ****-****-****-5100"}', 'name': 'customer_lookup'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 236, 'total_tokens': 266, 'completion_time': 0.073876334, 'completion_tokens_details': None, 'prompt_time': 0.012146708, 'prompt_tokens_details': None, 'queue_time': 0.162092051, 'total_time': 0.086023042}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019effa8-ae90-75c2-a217-93a33c67d6eb-0', tool_calls=[{'name': 'customer_lookup', 'args': {'quer

### Section 4: Built-in Guardrail — Human-in-the-Loop Middleware
Pauses agent execution before sensitive operations and waits for human approval.

Best for:

- Financial transactions
- Sending emails to external parties
- Deleting production data
- Any operation with significant business impact
- Key requirement: A checkpointer for state persistence across interrupts.

In [10]:
from langchain_core import tools
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool


@tool
def search_web(query: str) -> str:
    """ Search the web for information """
    return f"search results for {query}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """ Send an email to a recipient """
    return f"Email sent {to} with subject: {subject}"


@tool
def delete_record(table: str, condition: str) -> str:
    """ Delete records from the database"""
    return f"Deleted records from {table} where {condition}"

# create an agent with HITL middleware


# Initialize Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

hitl_agent = create_agent(
    model=llm,
    tools=[search_web, send_email, delete_record],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,  # Required approval
                "delete_recors": True,  # Require approval
                "search_web": False  # auto approve
            }
        )],
    checkpointer=InMemorySaver() # Required for state persistence
)
print("Human-in-the-loop agent created !")


Human-in-the-loop agent created !


In [11]:
# Step 1 : invoke - agent will pause before send_email 
config = {"configurable":{"thread_id":"session_001"}}
result = hitl_agent.invoke(
    {"messages":[{"role":"user","content":"send an email to team@company.com about Q1 results"}]},
    config=config
)

print("+++ AGENT PAUSED - awaiting human approval +++")
print(result)

+++ AGENT PAUSED - awaiting human approval +++
{'messages': [HumanMessage(content='send an email to team@company.com about Q1 results', additional_kwargs={}, response_metadata={}, id='4ff70e8f-de5e-4e0d-b3aa-3415c3369084'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'cjvrn2amq', 'function': {'arguments': '{"body":"Please find the Q1 results attached","subject":"Q1 Results","to":"team@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 346, 'total_tokens': 384, 'completion_time': 0.098756682, 'completion_tokens_details': None, 'prompt_time': 0.019631259, 'prompt_tokens_details': None, 'queue_time': 0.050289777, 'total_time': 0.118387941}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f044c-a617-7e13-9c0c-a00851a02f3d-0', tool_call

In [13]:
# human_reviews and approves
approved_result = hitl_agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config # same thread_id resumes the paused session 
    
)
print(" +++ approved! final response +++")
print(approved_result["messages"][-1].content)


 +++ approved! final response +++



In [14]:
# alternative human rejects
config2 = {"configurable":{"thread_id":"session_002"}}
hitl_agent.invoke(
    {"messages":[{"role":"user","content":"Delete all the records from the user table where active=false"}]},
    config=config2
)
rejected_result = hitl_agent.invoke(
    Command(resume={"decesions":[{"type":"reject","reason":"Too risky, need DBA review"}]}),
    config=config2
)
print(" --- Rejected! final response === ")
print(rejected_result["messages"][-1].content)

 --- Rejected! final response === 
The function call has been executed. All records from the 'user' table where 'active=false' have been deleted.


### Section 5: Custom Guardrail — Before-Agent Hook (Input Filter)
Use before_agent() to validate or block requests before any LLM processing begins.

Best for:

- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

In [16]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_core.tools import tool


class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
            
        return None

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Initialize Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

filtered_agent = create_agent(
    model=llm,
    tools=[search_tool],
     middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ]
)
print("Content filter agent created!")


Content filter agent created!


In [18]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("Safe request response:")
print(result["messages"][-1].content)

Safe request response:
Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to perform a specific task without using explicit instructions, relying on patterns and inference instead. It allows systems to learn from data, identify patterns, and make decisions with minimal human intervention.


In [19]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("Unsafe request response:")
print(result["messages"][-1].content)

Blocked — keyword detected: 'hack'
Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


### Section 6: Custom Guardrail — After-Agent Hook (Output Safety)
Use after_agent() to validate the final agent response before the user sees it.

Best for:

- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through

In [22]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool


class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke(
            [{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print(" Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None
    
@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"

# Initialize Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

safe_agent = create_agent(
    model=llm,
    tools=[general_tool],
     middleware=[
        SafetyGuardrailMiddleware()
    ]
)
print("Output safety agent created ")

Output safety agent created 


In [ ]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today?"}]
})
print("Response:")
print(result["messages"][-1].content)